In [ ]:
import time
import requests
import pandas as pd
from pathlib import Path

# Paths
#main_dir = Path("/Users/ga0028vi/Documents/MCcharactertization/ROSMAP/Top_sets/Set309")
main_dir = Path("C:/Users/gabri/Documents/Data_projects/MCcharactertization/ROSMAP/Top_sets/Set309")

set_data_dir = main_dir / "_set_data"

# Reference files live here:
hvg_dir = set_data_dir

out_dir = main_dir / "STRING_AGM_analysis" / "Annotations"
out_dir.mkdir(parents=True, exist_ok=True)

first_pass_dir = out_dir / "ref_pass1_ensembl"
missing_dir = out_dir / "ref_missing_genes"
final_dir = out_dir / "ref_final_annotations"

first_pass_dir.mkdir(parents=True, exist_ok=True)
missing_dir.mkdir(parents=True, exist_ok=True)
final_dir.mkdir(parents=True, exist_ok=True)

In [2]:
# 1. Ensembl query function
def query_ensembl_gene_biotypes(
    genes,
    species="homo_sapiens",
    batch_size=500,
    sleep_time=0.2
):
    """
    Query Ensembl REST API for gene biotypes using gene symbols.

    Parameters
    ----------
    genes : list-like
        Gene symbols, e.g. ["ISG15", "AL445685.1"].
    species : str
        Ensembl species name. For human, use "homo_sapiens".
    batch_size : int
        Number of symbols per API request.
    sleep_time : float
        Pause between requests to be polite to the API.

    Returns
    -------
    pd.DataFrame
        Annotation table with one row per returned gene.
    """

    genes = (
        pd.Series(genes)
        .dropna()
        .astype(str)
        .str.strip()
        .unique()
        .tolist()
    )

    url = f"https://rest.ensembl.org/lookup/symbol/{species}"

    headers = {
        "Content-Type": "application/json",
        "Accept": "application/json"
    }

    all_rows = []

    for i in range(0, len(genes), batch_size):
        batch = genes[i:i + batch_size]

        payload = {
            "symbols": batch
        }

        response = requests.post(
            url,
            headers=headers,
            json=payload,
            timeout=60
        )

        if response.status_code != 200:
            print(f"Batch {i // batch_size + 1} failed with status {response.status_code}")
            print(response.text[:500])
            continue

        data = response.json()

        for query_symbol, hit in data.items():
            if hit is None:
                continue

            all_rows.append({
                "gene_query": query_symbol,
                "ensembl_gene_id": hit.get("id"),
                "display_name": hit.get("display_name"),
                "description": hit.get("description"),
                "biotype": hit.get("biotype"),
                "object_type": hit.get("object_type"),
                "source": hit.get("source"),
                "seq_region_name": hit.get("seq_region_name"),
                "start": hit.get("start"),
                "end": hit.get("end"),
                "strand": hit.get("strand")
            })

        print(f"Queried {min(i + batch_size, len(genes))}/{len(genes)} genes")
        time.sleep(sleep_time)

    return pd.DataFrame(all_rows)

In [ ]:
# 2. Read reference gene TXT files
def read_reference_gene_files(hvg_dir):
    """
    Read all *_ref_genes.txt files into a dictionary.

    Returns
    -------
    dict
        Dictionary where keys are major cell type names and values are
        unique gene lists.
    """

    hvg_dir = Path(hvg_dir)
    ref_genes_by_major = {}

    for txt in hvg_dir.glob("*_ref_genes.txt"):
        major = txt.stem.replace("_ref_genes", "")

        seen = set()
        genes = []

        with open(txt, "r", encoding="utf-8") as f:
            for line in f:
                gene = line.strip()

                if not gene:
                    continue

                gene = gene.upper()

                if gene not in seen:
                    seen.add(gene)
                    genes.append(gene)

        ref_genes_by_major[major] = genes

    print("Reference gene files loaded:")
    for major, genes in sorted(ref_genes_by_major.items()):
        print(f"  {major}: {len(genes)} genes")

    return ref_genes_by_major

ref_genes_by_major = read_reference_gene_files(hvg_dir)

Reference gene files loaded:
  Astrocytes: 2213 genes
  Excitatory: 2455 genes
  Immune: 2247 genes
  Inhibitory: 1929 genes
  OPCs: 2124 genes
  Oligodendrocytes: 2391 genes
  Vasculature: 2743 genes


In [ ]:
# 3. First-pass Ensembl annotation for each major cell type

for major, genes in ref_genes_by_major.items():

    print("\n" + "=" * 80)
    print(f"First-pass Ensembl annotation: {major}")
    print("=" * 80)

    # Make one dataframe containing all original reference genes
    ref_df = pd.DataFrame({
        "major_cell_type": major,
        "gene_og": genes
    })

    # Query Ensembl using the original gene symbols
    annot_df = query_ensembl_gene_biotypes(genes)

    # Rename gene_query so we can merge it against gene_og
    annot_df = annot_df.rename(columns={"gene_query": "gene_og"})

    # Merge annotation back to full reference gene list
    ref_annotated_df = ref_df.merge(
        annot_df,
        on="gene_og",
        how="left"
    )

    ref_annotated_df["annotation_found"] = ref_annotated_df["ensembl_gene_id"].notna()

    # At this stage, before HGNC correction, renamed gene is just the original gene
    ref_annotated_df["gene_renamed"] = ref_annotated_df["gene_og"]
    ref_annotated_df["gene_name_changed_by_hgnc"] = False

    # Reorder useful columns
    first_cols = [
        "major_cell_type",
        "gene_og",
        "gene_renamed",
        "gene_name_changed_by_hgnc",
        "annotation_found"
    ]

    other_cols = [c for c in ref_annotated_df.columns if c not in first_cols]
    ref_annotated_df = ref_annotated_df[first_cols + other_cols]

    # Save first-pass annotation
    outfile = first_pass_dir / f"{major}_ref_genes_first_pass_ensembl.csv"
    ref_annotated_df.to_csv(outfile, index=False)
    print(f"Saved first-pass annotation: {outfile}")

    # Save missing genes for HGNC
    missing_df = (
        ref_annotated_df
        .loc[~ref_annotated_df["annotation_found"], ["gene_og"]]
        .drop_duplicates()
        .sort_values("gene_og")
    )

    miss_file = missing_dir / f"{major}_ref_genes_missing_for_hgnc.csv"
    missing_df.to_csv(miss_file, index=False)
    print(f"Saved missing genes for HGNC: {miss_file}")


First-pass Ensembl annotation: Astrocytes
Queried 500/2213 genes
Queried 1000/2213 genes
Queried 1500/2213 genes
Queried 2000/2213 genes
Queried 2213/2213 genes
Saved first-pass annotation: C:\Users\gabri\Documents\Data_projects\MCcharactertization\ROSMAP\Top_sets\Set309\STRING_AGM_analysis\Annotations\ref_pass1_ensembl\Astrocytes_ref_genes_first_pass_ensembl.csv
Saved missing genes for HGNC: C:\Users\gabri\Documents\Data_projects\MCcharactertization\ROSMAP\Top_sets\Set309\STRING_AGM_analysis\Annotations\ref_missing_genes\Astrocytes_ref_genes_missing_for_hgnc.csv

First-pass Ensembl annotation: Excitatory
Queried 500/2455 genes
Queried 1000/2455 genes
Queried 1500/2455 genes
Queried 2000/2455 genes
Queried 2455/2455 genes
Saved first-pass annotation: C:\Users\gabri\Documents\Data_projects\MCcharactertization\ROSMAP\Top_sets\Set309\STRING_AGM_analysis\Annotations\ref_pass1_ensembl\Excitatory_ref_genes_first_pass_ensembl.csv
Saved missing genes for HGNC: C:\Users\gabri\Documents\Data_pr

### Cleaning up the gene lists

In [ ]:
# 4. read HGNC symbol checker output

def read_hgnc_symbol_checker_output(hgnc_file):
    """
    Read HGNC symbol checker output and keep only useful symbol mappings.

    Expected columns:
    - Input
    - Approved symbol
    """

    hgnc_df = pd.read_csv(hgnc_file, skiprows=1)

    # Remove weird HGNC empty row where Input is 0
    hgnc_df = hgnc_df[hgnc_df["Input"].astype(str) != "0"].copy()

    # Keep only rows where HGNC gives an approved replacement
    hgnc_df = hgnc_df[
        hgnc_df["Input"].notna() &
        hgnc_df["Approved symbol"].notna()
    ].copy()

    hgnc_df["Input"] = hgnc_df["Input"].astype(str).str.strip().str.upper()
    hgnc_df["Approved symbol"] = hgnc_df["Approved symbol"].astype(str).str.strip().str.upper()

    hgnc_df = hgnc_df[["Input", "Approved symbol"]].drop_duplicates()

    return hgnc_df

In [ ]:
# 5. Biotype categories

lncRNA_biotypes = {
    "lncRNA",
    "lincRNA",
    "antisense",
    "sense_intronic",
    "sense_overlapping",
    "processed_transcript",
    "3prime_overlapping_ncRNA",
    "macro_lncRNA",
    "bidirectional_promoter_lncRNA",
    "non_coding"
}

small_ncRNA_biotypes = {
    "miRNA",
    "snRNA",
    "snoRNA",
    "rRNA",
    "misc_RNA",
    "scaRNA",
    "scRNA",
    "ribozyme",
    "sRNA",
    "vault_RNA",
    "Mt_rRNA",
    "Mt_tRNA"
}

ncRNA_biotypes = lncRNA_biotypes.union(small_ncRNA_biotypes)

In [ ]:
# 6. Apply HGNC corrections and rerun Ensembl for each ref gene list

annot_cols = [
    "ensembl_gene_id",
    "display_name",
    "description",
    "biotype",
    "object_type",
    "source",
    "seq_region_name",
    "start",
    "end",
    "strand"
]

final_ref_dfs = []

for first_pass_file in sorted(first_pass_dir.glob("*_ref_genes_first_pass_ensembl.csv")):

    major = first_pass_file.name.replace("_ref_genes_first_pass_ensembl.csv", "")

    print("\n" + "=" * 80)
    print(f"Final annotation with HGNC correction: {major}")
    print("=" * 80)

    hgnc_file = missing_dir / f"{major}_HGNCcheck.csv"

    if not hgnc_file.exists():
        print(f"No HGNC file found for {major}, skipping HGNC correction.")
        print(f"Expected: {hgnc_file}")

        ref_df = pd.read_csv(first_pass_file)
        ref_df["is_lncRNA"] = ref_df["biotype"].isin(lncRNA_biotypes)
        ref_df["is_any_ncRNA"] = ref_df["biotype"].isin(ncRNA_biotypes)
        ref_df["is_protein_coding"] = ref_df["biotype"].eq("protein_coding")

        final_df = ref_df.copy()

    else:
        ref_df = pd.read_csv(first_pass_file)
        hgnc_df = read_hgnc_symbol_checker_output(hgnc_file)

        # Make sure gene_renamed starts as original gene symbol
        ref_df["gene_renamed"] = ref_df["gene_og"]

        # Merge HGNC Approved symbol onto original gene names
        ref_df = ref_df.merge(
            hgnc_df,
            left_on="gene_renamed",
            right_on="Input",
            how="left"
        )

        # Replace gene_renamed with HGNC-approved symbol where available
        ref_df.loc[
            ref_df["Approved symbol"].notna(),
            "gene_renamed"
        ] = ref_df.loc[
            ref_df["Approved symbol"].notna(),
            "Approved symbol"
        ]

        # Mark genes changed by HGNC
        ref_df["gene_name_changed_by_hgnc"] = (
            ref_df["gene_og"] != ref_df["gene_renamed"]
        )

        # Drop temporary HGNC columns
        ref_df = ref_df.drop(columns=["Input", "Approved symbol"])

        # Genes to rerun:
        # failed first-pass Ensembl annotation,
        # but got an HGNC-updated symbol
        genes_to_rerun = (
            ref_df.loc[
                (~ref_df["annotation_found"]) &
                (ref_df["gene_name_changed_by_hgnc"]),
                "gene_renamed"
            ]
            .dropna()
            .drop_duplicates()
            .tolist()
        )

        print(f"Genes to re-query after HGNC correction: {len(genes_to_rerun)}")
        print(genes_to_rerun[:10])

        if len(genes_to_rerun) > 0:
            rerun_annot_df = query_ensembl_gene_biotypes(genes_to_rerun)

            if len(rerun_annot_df) > 0:
                rerun_annot_df = rerun_annot_df.rename(
                    columns={"gene_query": "gene_renamed"}
                )

                # Merge rerun Ensembl annotations back by HGNC-renamed symbol
                final_df = ref_df.merge(
                    rerun_annot_df[["gene_renamed"] + annot_cols],
                    on="gene_renamed",
                    how="left",
                    suffixes=("", "_rerun")
                )

                # Use rerun annotation where available
                for col in annot_cols:
                    rerun_col = f"{col}_rerun"

                    if rerun_col in final_df.columns:
                        final_df[col] = final_df[rerun_col].combine_first(final_df[col])
                        final_df = final_df.drop(columns=[rerun_col])

            else:
                final_df = ref_df.copy()

        else:
            final_df = ref_df.copy()

        # Update annotation_found after rerun
        final_df["annotation_found"] = final_df["ensembl_gene_id"].notna()

        # Add ncRNA / protein-coding flags
        final_df["is_lncRNA"] = final_df["biotype"].isin(lncRNA_biotypes)
        final_df["is_any_ncRNA"] = final_df["biotype"].isin(ncRNA_biotypes)
        final_df["is_protein_coding"] = final_df["biotype"].eq("protein_coding")

    # Reorder columns
    cols_order = [
        "major_cell_type",
        "gene_og",
        "gene_renamed",
        "gene_name_changed_by_hgnc",
        "ensembl_gene_id",
        "display_name",
        "description",
        "biotype",
        "is_lncRNA",
        "is_any_ncRNA",
        "is_protein_coding",
        "object_type",
        "source",
        "seq_region_name",
        "start",
        "end",
        "strand",
        "annotation_found"
    ]

    final_df = final_df[cols_order].copy()

    # Save final per-major-cell-type annotation file
    outfile = final_dir / f"{major}_ref_genes_final_annotations.csv"
    final_df.to_csv(outfile, index=False)

    print(f"Saved final annotation: {outfile}")
    print(final_df["annotation_found"].value_counts(dropna=False))
    print(final_df["biotype"].value_counts(dropna=False).head(10))

    final_ref_dfs.append(final_df)


Final annotation with HGNC correction: Astrocytes
No HGNC file found for Astrocytes, skipping HGNC correction.
Expected: C:\Users\gabri\Documents\Data_projects\MCcharactertization\ROSMAP\Top_sets\Set309\STRING_AGM_analysis\Annotations\ref_missing_genes\Astrocytes_HGNCcheck.csv
Saved final annotation: C:\Users\gabri\Documents\Data_projects\MCcharactertization\ROSMAP\Top_sets\Set309\STRING_AGM_analysis\Annotations\ref_final_annotations\Astrocytes_ref_genes_final_annotations.csv
annotation_found
True     1950
False     263
Name: count, dtype: int64
biotype
protein_coding    1827
NaN                263
lncRNA             122
ribozyme             1
Name: count, dtype: int64

Final annotation with HGNC correction: Excitatory
No HGNC file found for Excitatory, skipping HGNC correction.
Expected: C:\Users\gabri\Documents\Data_projects\MCcharactertization\ROSMAP\Top_sets\Set309\STRING_AGM_analysis\Annotations\ref_missing_genes\Excitatory_HGNCcheck.csv
Saved final annotation: C:\Users\gabri\Doc

In [ ]:
# 7. Save combined final reference annotation file

combined_ref_df = pd.concat(final_ref_dfs, ignore_index=True)

combined_outfile = final_dir / "ALL_ref_genes_final_annotations.csv"
combined_ref_df.to_csv(combined_outfile, index=False)

print("\nSaved combined reference annotation file:")
print(combined_outfile)

print("\nAnnotation found:")
print(combined_ref_df["annotation_found"].value_counts(dropna=False))

print("\nBiotypes:")
print(combined_ref_df["biotype"].value_counts(dropna=False).head(20))

print("\nlncRNA counts by major cell type:")
print(
    combined_ref_df
    .groupby("major_cell_type")["is_lncRNA"]
    .agg(["sum", "count", "mean"])
    .sort_values("mean", ascending=False)
)


Saved combined reference annotation file:
C:\Users\gabri\Documents\Data_projects\MCcharactertization\ROSMAP\Top_sets\Set309\STRING_AGM_analysis\Annotations\ref_final_annotations\ALL_ref_genes_final_annotations.csv

Annotation found:
annotation_found
True     11638
False     1721
Name: count, dtype: int64

Biotypes:
biotype
protein_coding    10760
NaN                1721
lncRNA              866
TR_C_gene             6
IG_C_gene             5
ribozyme              1
Name: count, dtype: int64

lncRNA counts by major cell type:
                  sum  count      mean
major_cell_type                       
OPCs              193   2124  0.090866
Inhibitory        173   1929  0.089684
Oligodendrocytes  163   2391  0.068172
Immune            124   2247  0.055185
Astrocytes        122   2213  0.055129
Excitatory         91   2455  0.037067


fishers

In [19]:
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
import pandas as pd
from pathlib import Path


# ------------------------------------------------------------
# 0. Input files
# ------------------------------------------------------------

# AGM annotation file
agm_file = out_dir / "AGM_gene_list_long_with_biotypes_HGNC_rescued.csv"

# Combined reference annotation file from previous step
ref_file = final_dir / "ALL_ref_genes_final_annotations.csv"

agm_df = pd.read_csv(agm_file)
ref_df = pd.read_csv(ref_file)

In [51]:
agm_df["is_unannotated"] = ~agm_df["annotation_found"].fillna(False).astype(bool)
ref_df["is_unannotated"] = ~ref_df["annotation_found"].fillna(False).astype(bool)

In [ ]:
# 1. Pick category to test
#
#category = "is_lncRNA"
#category = "is_any_ncRNA"
#category = "gene_name_changed_by_hgnc"
#category = "annotation_found"
category = "is_unannotated"

In [ ]:
# 2. Extract major cell type from AGM_ID

agm_df["major_cell_type"] = agm_df["AGM_ID"].str.split("_").str[0]

print(agm_df["major_cell_type"].value_counts())
print(ref_df["major_cell_type"].value_counts())

major_cell_type
Excitatory          1093
Inhibitory           670
Immune               511
Oligodendrocytes     228
Astrocytes            98
OPCs                  76
Name: count, dtype: int64
major_cell_type
Excitatory          2455
Oligodendrocytes    2391
Immune              2247
Astrocytes          2213
OPCs                2124
Inhibitory          1929
Name: count, dtype: int64


In [ ]:
# 3. Make sure category is boolean

agm_df[category] = agm_df[category].fillna(False).astype(bool)
ref_df[category] = ref_df[category].fillna(False).astype(bool)

In [68]:
ref_df

,major_cell_type,gene_og,gene_renamed,gene_name_changed_by_hgnc,ensembl_gene_id,display_name,description,biotype,is_lncRNA,is_any_ncRNA,is_protein_coding,object_type,source,seq_region_name,start,end,strand,annotation_found,is_unannotated
0,Astrocytes,ABCA10,ABCA10,False,ENSG00000154263,ABCA10,ATP binding cassette subfamily A member 10 [So...,protein_coding,False,False,True,Gene,ensembl_havana,17,69147214.0,69244846.0,-1.0,True,False
1,Astrocytes,ABCA6,ABCA6,False,ENSG00000154262,ABCA6,ATP binding cassette subfamily A member 6 [Sou...,protein_coding,False,False,True,Gene,ensembl_havana,17,69074255.0,69143342.0,-1.0,True,False
2,Astrocytes,ABCA8,ABCA8,False,ENSG00000141338,ABCA8,ATP binding cassette subfamily A member 8 [Sou...,protein_coding,False,False,True,Gene,ensembl_havana,17,68866052.0,68955470.0,-1.0,True,False
3,Astrocytes,ABCA9,ABCA9,False,ENSG00000154258,ABCA9,ATP binding cassette subfamily A member 9 [Sou...,protein_coding,False,False,True,Gene,ensembl_havana,17,68974488.0,69061087.0,-1.0,True,False
4,Astrocytes,ABCB1,ABCB1,False,ENSG00000085563,ABCB1,ATP binding cassette subfamily B member 1 [Sou...,protein_coding,False,False,True,Gene,ensembl_havana,7,87501989.0,87713379.0,-1.0,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13354,OPCs,ZNF831,ZNF831,False,ENSG00000124203,ZNF831,zinc finger protein 831 [Source:HGNC Symbol;Ac...,protein_coding,False,False,True,Gene,ensembl_havana,20,59123381.0,59259113.0,1.0,True,False
13355,OPCs,ZNF90,ZNF90,False,ENSG00000213988,ZNF90,zinc finger protein 90 [Source:HGNC Symbol;Acc...,protein_coding,False,False,True,Gene,ensembl_havana,19,20077933.0,20132733.0,1.0,True,False
13356,OPCs,ZNRF3-AS1,ZNRF3-AS1,False,ENSG00000177993,ZNRF3-AS1,ZNRF3 antisense RNA 1 [Source:HGNC Symbol;Acc:...,lncRNA,True,True,False,Gene,havana,22,29024999.0,29031533.0,-1.0,True,False
13357,OPCs,ZRANB3,ZRANB3,False,ENSG00000121988,ZRANB3,zinc finger RANBP2-type containing 3 [Source:H...,protein_coding,False,False,True,Gene,ensembl_havana,2,135136916.0,135531228.0,-1.0,True,False


In [ ]:
# 4. Fisher exact test per AGM using matching major-cell-type background

gene_col = "gene_renamed"

results = []

for agm_id in agm_df["AGM_ID"].dropna().unique():

    # Get major cell type for this AGM
    major = agm_df.loc[
        agm_df["AGM_ID"] == agm_id,
        "major_cell_type"
    ].iloc[0]

    # AGM genes
    agm_genes = set(
        agm_df.loc[
            agm_df["AGM_ID"] == agm_id,
            gene_col
        ]
        .dropna()
        .astype(str)
        .str.upper()
        .unique()
    )

    # Matching reference/background genes for this major cell type
    bg_df = ref_df.loc[
        ref_df["major_cell_type"] == major
    ].copy()

    bg_genes = set(
        bg_df[gene_col]
        .dropna()
        .astype(str)
        .str.upper()
        .unique()
    )

    # Keep only AGM genes that are actually present in the matching background
    agm_genes_in_bg = agm_genes.intersection(bg_genes)

    # Background excluding this AGM
    other_genes = bg_genes - agm_genes_in_bg

    # Category status in this background
    gene_status = (
        bg_df[[gene_col, category]]
        .dropna(subset=[gene_col])
        .assign(**{gene_col: lambda x: x[gene_col].astype(str).str.upper()})
        .drop_duplicates(gene_col)
        .set_index(gene_col)[category]
        .to_dict()
    )

    category_in_agm = sum(gene_status[g] for g in agm_genes_in_bg)
    not_category_in_agm = len(agm_genes_in_bg) - category_in_agm

    category_outside = sum(gene_status[g] for g in other_genes)
    not_category_outside = len(other_genes) - category_outside

    table = [
        [category_in_agm, not_category_in_agm],
        [category_outside, not_category_outside]
    ]

    odds_ratio, pval = fisher_exact(table, alternative="greater")

    results.append({
        "AGM_ID": agm_id,
        "major_cell_type": major,
        "category": category,

        "n_AGM_genes_original": len(agm_genes),
        "n_AGM_genes_in_background": len(agm_genes_in_bg),
        "n_AGM_genes_not_in_background": len(agm_genes - bg_genes),

        "n_background_genes": len(bg_genes),
        "n_background_genes_outside_AGM": len(other_genes),

        "n_category_in_AGM": category_in_agm,
        "n_not_category_in_AGM": not_category_in_agm,

        "n_category_outside_AGM": category_outside,
        "n_not_category_outside_AGM": not_category_outside,

        "prop_category_in_AGM": category_in_agm / len(agm_genes_in_bg),
        "prop_category_outside_AGM": category_outside / len(other_genes),
        "prop_category_background_total": (
            (category_in_agm + category_outside) / len(bg_genes)
        ),

        "odds_ratio": odds_ratio,
        "pval": pval
    })

res_df = pd.DataFrame(results)

res_df["FDR"] = multipletests(
    res_df["pval"],
    method="fdr_bh"
)[1]

res_df = res_df.sort_values(["FDR", "pval"])

res_df

,AGM_ID,major_cell_type,category,n_AGM_genes_original,n_AGM_genes_in_background,n_AGM_genes_not_in_background,n_background_genes,n_background_genes_outside_AGM,n_category_in_AGM,n_not_category_in_AGM,n_category_outside_AGM,n_not_category_outside_AGM,prop_category_in_AGM,prop_category_outside_AGM,prop_category_background_total,odds_ratio,pval,FDR
1,Excitatory_ExcL45RORBGABRG1_Cluster_7,Excitatory,is_unannotated,78,77,1,2455,2378,23,54,187,2191,0.298701,0.078638,0.085540,4.990394,3.318969e-08,7.965525e-07
19,Oligodendrocytes_Oli_Cluster_12,Oligodendrocytes,is_unannotated,92,87,5,2391,2304,22,65,291,2013,0.252874,0.126302,0.130908,2.341316,1.279690e-03,1.535628e-02
7,Excitatory_ExcL35RORBPLCH1_Cluster_14,Excitatory,is_unannotated,221,217,4,2455,2238,30,187,180,2058,0.138249,0.080429,0.085540,1.834225,4.306565e-03,3.445252e-02
16,Inhibitory_InhLAMP5NRG1Rosehip_Cluster_7,Inhibitory,is_unannotated,93,91,2,1929,1838,22,69,293,1545,0.241758,0.159412,0.163297,1.681258,3.114840e-02,1.868904e-01
9,Excitatory_ExcL34RORBCUX2_Cluster_6,Excitatory,is_unannotated,104,100,4,2455,2355,13,87,197,2158,0.130000,0.083652,0.085540,1.636852,8.036334e-02,3.789007e-01
0,Excitatory_ExcL45RORBGABRG1_Cluster_0,Excitatory,is_unannotated,144,143,1,2455,2312,17,126,193,2119,0.118881,0.083478,0.085540,1.481331,9.791403e-02,3.789007e-01
8,Excitatory_ExcL6CT_Cluster_14,Excitatory,is_unannotated,41,39,2,2455,2416,6,33,204,2212,0.153846,0.084437,0.085540,1.971480,1.105127e-01,3.789007e-01
2,Excitatory_ExcL45RORBGABRG1_Cluster_8,Excitatory,is_unannotated,63,61,2,2455,2394,8,53,202,2192,0.131148,0.084378,0.085540,1.637960,1.452089e-01,4.356268e-01
15,Inhibitory_InhLAMP5NRG1Rosehip_Cluster_6,Inhibitory,is_unannotated,260,252,8,1929,1677,47,205,268,1409,0.186508,0.159809,0.163297,1.205369,1.638675e-01,4.369800e-01
11,Inhibitory_InhG7_Cluster_10,Inhibitory,is_unannotated,57,56,1,1929,1873,12,44,303,1570,0.214286,0.161773,0.163297,1.413141,1.907904e-01,4.578969e-01
